# Baseline - word matching

This notebook explores building a baseline that matches the taxonomi words with the text in the job ads.

There are two versions:
1. The first version matches the untouched skill list from the taxonomi (skill + ev. disambiguation) to the descriptions in the ads.
2. The second version alters the taxonomi, strictly keeping the skill before the comma, and removes the specifications/disambiguations that come after. Example: `"Linux, operativsystem"` turns into `"Linux"`. This allows the matching to the text to catch more hits.

Result:  
The second version collects more skills and will therefore be used as baseline model.

The last step is the validation of the baseline on the labeled validation set. Many missed skills from the baseline will be included in the next rules for the next approach to skill extraction.

In [1]:
import json
import pandas as pd
import re
from collections import Counter

from src.skills import TIER1_PHRASES, TIER2_EXACT_CASE, TIER3_CUSTOM
from src.evaluation import (
    build_alias_to_canonical, 
    normalize_all_ads, 
    calculate_micro_metrics, 
    calculate_macro_metrics, 
    create_error_analysis,
    summarize_errors,
    errors_to_dataframe,
    build_canonical_to_aliases,
    review_ad
)


with open("../data/train_set.json", "r", encoding="utf-8") as f:
    ads_df = pd.DataFrame(json.load(f))

with open("../data/taxonomi_2511.json", "r", encoding="utf-8") as f:
    skills_df = pd.DataFrame(json.load(f))

In [2]:
# help function to extract skills from text
def extract_skills(text, skills):
    text = text.lower()
    return [
        skill for skill in skills
        if re.search(r"\b" + re.escape(skill.lower()) + r"\b", text) # this allows me to match a skill to an entire word
    ]

## Version 1

In [3]:
skills_full = skills_df["skill"].tolist()

ads_df["baseline_full"] = ads_df["description"].apply(
    lambda text: extract_skills(text, skills_full)
)

In [4]:
baseline_full_skills = ads_df["baseline_full"].explode()
baseline_full_count = baseline_full_skills.value_counts()

## Version 2

In [5]:
# removing specifications/disambiguations from the comma onwards
skills_short = (
    skills_df["skill"]
    .str.split(",").str[0]
    .str.strip()
    .drop_duplicates()
    .tolist()
)

ads_df["baseline_short"] = ads_df["description"].apply(
    lambda text: extract_skills(text, skills_short)
)

In [6]:
baseline_short_skills = ads_df["baseline_short"].explode()
baseline_short_count = baseline_short_skills.value_counts()

## Comparing the two versions

In [7]:
ads_df[["baseline_full", "baseline_short"]]

,baseline_full,baseline_short
0,[],[]
1,[],[SAP]
2,[],[]
3,[],[]
4,[],[]
...,...,...
2601,[],[]
2602,[],[SAP]
2603,[],[]
2604,[],"[Visual Studio, LINQ, ASP.NET MVC]"


In [8]:
comparison = pd.DataFrame({
    "baseline_full": baseline_full_count,
    "baseline_short": baseline_short_count
}).fillna(0)

comparison

,baseline_full,baseline_short
API-design,12.0,12.0
ASP.NET MVC,0.0,3.0
Agila arbetsmetoder,21.0,21.0
Android,0.0,12.0
Apache Kafka,0.0,9.0
...,...,...
Visual Studio Code,0.0,7.0
Webbutveckling,2.0,2.0
Windows,0.0,65.0
Windows Server,0.0,31.0


## Validation on baseline version 2

In [9]:
with open("../data/labeled_val_set.json", "r", encoding="utf-8") as file:
    val_df = pd.DataFrame(json.load(file))

print("Ads in validation set: ", len(val_df))
val_df.head(3)

Ads in validation set:  48


,id,description,skills,spans
0,31398837,Om företaget Siemens Energy möter den ökade ef...,"[Alteryx-baserade dataintegrationslösning, Des...","[[652, 661, Snowflake], [692, 714, Snowflake-a..."
1,30714137,This position combines deep technical expertis...,"[Architecture Competencies, Automotive Afterma...","[[585, 615, Single Point of Contact (SPOC)], [..."
2,31553577,Etteplan är ett ingenjörsföretag med över 4 00...,"[DOORS, Kravanalys, Planering och dokumentatio...","[[764, 774, Kravanalys], [776, 791, kravnedbry..."


In [10]:
val_df = val_df.drop(columns="spans")
val_df = val_df.rename(columns={"skills": "gold_raw"})

val_df["pred_raw"] = val_df["description"].apply(lambda text: extract_skills(text, skills_short))

val_df.head()

,id,description,gold_raw,pred_raw
0,31398837,Om företaget Siemens Energy möter den ökade ef...,"[Alteryx-baserade dataintegrationslösning, Des...",[]
1,30714137,This position combines deep technical expertis...,"[Architecture Competencies, Automotive Afterma...",[]
2,31553577,Etteplan är ett ingenjörsföretag med över 4 00...,"[DOORS, Kravanalys, Planering och dokumentatio...",[]
3,31349669,Om oss På Techrytera AB börjar rekryteringen m...,"[AI, Azure, Microsoft 365, SharePoint-kompeten...",[Informationsarkitektur]
4,30472352,Om företaget Avaron AB är ett växande konsultb...,[Dokumentera resultat och presentera slutsatse...,[]


In [11]:
alias_to_canonical = build_alias_to_canonical(TIER1_PHRASES, TIER2_EXACT_CASE, TIER3_CUSTOM)

gold_raw = dict(zip(val_df["id"], val_df["gold_raw"]))
pred_raw = dict(zip(val_df["id"], val_df["pred_raw"]))

# convert all ads, for gold and predictions
gold = normalize_all_ads(gold_raw, alias_to_canonical)
pred = normalize_all_ads(pred_raw, alias_to_canonical)

In [12]:
micro_metrics = calculate_micro_metrics(gold, pred)
macro_metrics = calculate_macro_metrics(gold, pred)

comparison = pd.DataFrame({"micro": micro_metrics, "macro": macro_metrics})

comparison.head(3)

,micro,macro
precision,0.343,0.208
recall,0.013,0.015
f1,0.025,0.028


In [13]:
# 1. Most frequently missed skills
fn_counter = Counter(s for i in gold for s in gold[i] - pred[i])
print(fn_counter.most_common(20))

# 2. Coverage: share of gold skills that exist in the alias map at all
known = set(alias_to_canonical.values())
all_gold = [s for i in gold for s in gold[i]]
print("Gold skills covered by alias map:", sum(s in known for s in all_gold) / len(all_gold))

[('Swedish', 19), ('English', 18), ('Artificial Intelligence', 8), ('Python', 7), ('Microsoft 365', 6), ('Automation', 6), ('Data modelling', 5), ('Windows Azure', 5), ('Entra ID', 4), ('verifiering', 3), ('Kravhantering', 3), ('API', 3), ('informationssäkerhet', 3), ('Kubernetes', 3), ('Infrastructure as Code', 3), ('implementation', 3), ('Integrationer', 3), ('Configuration Management (CM)', 3), ('Agile', 3), ('lösningsdesign', 3)]
Gold skills covered by alias map: 0.24591947769314473


### Error analysis from validation

In [14]:
errors = create_error_analysis(gold, pred)

summary = summarize_errors(errors, alias_to_canonical)

In [15]:
print("How many misses are dictionary gaps:\n", summary["fn_types"]) 
print("\nThe baseline real mistakes:\n", summary["top_fp"])

How many misses are dictionary gaps:
 {'not in dictionary': 693, 'in dictionary, not matched': 214}

The baseline real mistakes:
 [('Projektledning', 7), ('SAP', 3), ('Logic', 2), ('Windows', 2), ('Scrum', 2), ('Change management', 2), ('R', 1), ('REST', 1), ('Linux', 1), ('Masterdata', 1), ('Kravhantering', 1)]


In [16]:
errs = errors_to_dataframe(errors)
errs.head(10) 

,id,false_positives,false_negatives,n_fp,n_fn
0,31273786,[],"[API, Architecture Decision Records, CDC, CORE...",0,63
1,30940958,"[Logic, R]","[Active Directory, Automate workflows, Azure F...",2,38
2,30703637,[],"[AI & LLM Engineering, AI models and algorithm...",0,35
3,31079160,[],"[AI-infrastruktur, Amazon Web Services, Ansibl...",0,34
4,31519605,"[Masterdata, SAP]","[Artificial Intelligence, BTP, Clean Core, Dri...",2,33
5,31157086,[REST],"[A/B testing, ARR, Dagster, Data modelling, Da...",1,32
6,31015022,[],"[AI coding tools, Building and productionizing...",0,28
7,30568658,[],"[AKS, Automation, Azure DevOps, Azure infrastr...",0,27
8,31209146,[Change management],"[AI-adoption, AI-governance, AI-strategi, Arti...",1,26
9,31564504,[],"[AI-plattform, AI-plattformar, AI-tjänster, Ar...",0,26


In [17]:
canonical_to_aliases = build_canonical_to_aliases(alias_to_canonical)
review_ad(31519605, val_df, gold, pred, canonical_to_aliases)

gold: ['Artificial Intelligence', 'BTP', 'Clean Core', 'Driva workshops', 'English', 'FI', 'Fiori', 'Integrationer', 'MM', 'Order-to-Cash', 'Output management', 'S/4HANA', 'S/4HANA-transformationer', 'SAP S/4HANA', 'SAP Sales/SD', 'SAP-miljöer', 'SAP-transaktioner', 'SD', 'Swedish', 'aATP', 'cutover', 'datamigrering', 'driftsättning', 'förvaltning', 'globala rollouts', 'hypercare', 'konfiguration', 'konfigurera och vidareutveckla SAP SD', 'krav', 'lösningsdesign', 'processharmonisering', 'rollouts', 'test']
pred: ['Masterdata', 'SAP']
